<a href="https://colab.research.google.com/github/mybright107/workflow_python/blob/main/BTAA_overlap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# BTAA Overlap Combine task — Google Colab version
# ============================================================
# Run this in a Colab notebook. It will:
#  1. Upload YOUR Input File
#  2. Process it
#  3. Download the resulting xlsx automatically in your Colab Notebook
# ============================================================

# --- Cell 1: install dependencies (Colab usually has these already) ---
!pip install -q openpyxl pandas

In [ ]:
# --- Cell 2: mount Google Drive and point to your CSV ---
from google.colab import drive
drive.mount('/content/drive')



In [ ]:
# Update this path to where your file is in Google Drive
INPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/[TYPE YOUR INPUT FILENAME].csv"

In [ ]:
# --- Cell 3: run the processing ---
import pandas as pd
import re
from difflib import SequenceMatcher
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font

OUTPUT_FILE = "[TYPE YOUR OUTPUT FILE NAME].xlsx"  #Update the file name
DATE_VALUE = "BTAA as of 06/19/2026"  #Update this data value in your local note
Q_VALUE = "2"  #Update this 916$q value per report (e.g. "3" or "18")

ABBREV = {
    "indiana university": "IU",
    "michigan state university": "MSU",
    "northwestern university": "NWU",
    "pennsylvania state university": "PSU",
    "purdue university": "PU",
    "rutgers university": "RU",
    "the ohio state university": "OSU",
    "university of california-los angeles": "UCLA",
    "university of chicago": "Chicago",
    "university of illinois": "UIUC",
    "university of iowa": "UI",
    "university of maryland": "MD",
    "university of michigan": "MI",
    "university of minnesota": "MINN",
    "university of nebraska-lincoln": "UN",
    "university of oregon": "UO",
    "university of southern california": "USC",
    "university of washington": "UW",
    "university of wisconsin-madison": "UW-M",
}

STOPWORDS = {"university", "univ", "the", "of", "at", "college", "campus"}
MATCH_THRESHOLD = 0.7


def normalize(name):
    name = str(name).lower()
    name = re.split(r"\bat\b", name)[0]  # drop "at <campus>" suffix
    name = re.sub(r"[^a-z0-9\- ]", " ", name)
    tokens = [t for t in name.split() if t not in STOPWORDS]
    return " ".join(tokens)


NORM_ABBREV = {normalize(k): v for k, v in ABBREV.items()}


def match_library(lib_name):
    norm = normalize(lib_name)
    best_score, best_abbrev = 0.0, None
    for key, abbrev in NORM_ABBREV.items():
        score = SequenceMatcher(None, norm, key).ratio()
        if score > best_score:
            best_score, best_abbrev = score, abbrev
    return best_abbrev, best_score


_match_cache = {}


def cached_match(lib_name):
    if lib_name not in _match_cache:
        _match_cache[lib_name] = match_library(lib_name)
    return _match_cache[lib_name]


df = pd.read_csv(INPUT_FILE, dtype=str, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]
ctrl_col, lib_col, key_col = df.columns[0], df.columns[1], df.columns[2]

wb = Workbook()
ws = wb.active
ws.title = "Combined"

headers = ["001", "916$q", "916$s", "916$t", "Notes"]  #Update the outputfile header if needed
ws.append(headers)

yellow = PatternFill("solid", start_color="FFFF99", end_color="FFFF99")
orange = PatternFill("solid", start_color="FFCC99", end_color="FFCC99")
pink = PatternFill("solid", start_color="FFCCCC", end_color="FFCCCC")

n = len(df)
i = 0
out_row = 1  # header is row 1
while i < n:
    key_i = str(df.iloc[i][key_col]).strip()

    # collect all consecutive rows sharing this MatchKey
    j = i
    while j + 1 < n and str(df.iloc[j + 1][key_col]).strip() == key_i:
        j += 1
    group = df.iloc[i:j + 1]

    if len(group) > 1:
        note = []
        row_fill = None

        # library names from ALL rows in group, deduped + fuzzy matched
        libs = []
        for _, grow in group.iterrows():
            lib, score = cached_match(grow[lib_col])
            if score < MATCH_THRESHOLD:
                note.append(f"Low-confidence library match for '{grow[lib_col]}'")
            if lib:
                libs.append(lib)
        libs = sorted(set(libs))
        libs_str = "; ".join(libs)

        # find all control numbers ending in 3731 -- USC Alma IZ MMS ID ends with 3731
        ctrl_3731 = [str(grow[ctrl_col]).strip() for _, grow in group.iterrows()
                      if str(grow[ctrl_col]).strip().endswith("3731")]

        if len(ctrl_3731) == 0:
            row_fill = orange
            note.append("REVIEW: no control number in group ends in 3731")
            ws.append(["", Q_VALUE, libs_str, DATE_VALUE, "; ".join(note)])
            out_row += 1
            if row_fill:
                for c in range(1, len(headers) + 1):
                    ws.cell(row=out_row, column=c).fill = row_fill
        else:
            if any(s < MATCH_THRESHOLD for _, grow in group.iterrows()
                   for _, s in [cached_match(grow[lib_col])]):
                row_fill = pink
            for ctrl in ctrl_3731:
                ws.append([ctrl, Q_VALUE, libs_str, DATE_VALUE, "; ".join(note)])
                out_row += 1
                if row_fill:
                    for c in range(1, len(headers) + 1):
                        ws.cell(row=out_row, column=c).fill = row_fill

        i = j + 1
    else:
        row = df.iloc[i]
        ctrl1 = str(row[ctrl_col]).strip()
        lib1, score1 = cached_match(row[lib_col])
        note = ["UNMATCHED: no adjacent row with same MatchKey - not combined"]
        row_fill = yellow
        if score1 < MATCH_THRESHOLD:
            note.append(f"Low-confidence library match for '{row[lib_col]}'")

        ws.append([ctrl1, Q_VALUE, lib1, DATE_VALUE, "; ".join(note)])
        out_row += 1
        if row_fill:
            for c in range(1, len(headers) + 1):
                ws.cell(row=out_row, column=c).fill = row_fill

        i += 1

widths = [22, 8, 35, 24, 60]
for idx, w in enumerate(widths, start=1):
    ws.column_dimensions[ws.cell(row=1, column=idx).column_letter].width = w

arial = Font(name="Arial")
arial_bold = Font(name="Arial", bold=True)
for c in range(1, len(headers) + 1):
    ws.cell(row=1, column=c).font = arial_bold
for r in range(2, out_row + 1):
    for c in range(1, len(headers) + 1):
        ws.cell(row=r, column=c).font = arial

legend = wb.create_sheet("Legend")
legend.append(["Color", "Meaning"])
legend["A1"].font = Font(name="Arial", bold=True)
legend["B1"].font = Font(name="Arial", bold=True)

legend.append(["Yellow", "Row had no adjacent row with a matching MatchKey - left as a single, uncombined row. Please review."])
legend["A2"].fill = yellow
legend.append(["Orange", "No control number in the matched group ends in '3731' - the 001 field is blank and needs manual review."])
legend["A3"].fill = orange
legend.append(["Pink", "At least one library name in the matched group could not be confidently matched to the abbreviation table - check the 916$s value."])
legend["A4"].fill = pink

for row in legend.iter_rows():
    for cell in row:
        cell.font = Font(name="Arial")
legend.column_dimensions["A"].width = 12
legend.column_dimensions["B"].width = 90

wb.save(OUTPUT_FILE)
print(f"Saved {OUTPUT_FILE}")


In [ ]:
# --- Cell 4: save the result to Google Drive ---
import shutil
drive_output = "/content/drive/MyDrive/Colab Notebooks/[TYPE YOUR OURPUT FILE NAME].xlsx" #Updae your output file name
shutil.copy(OUTPUT_FILE, drive_output)
print(f"Saved to {drive_output}")